# Saúde — Indicadores de Diabetes: EDA + Mineração + Apoio à Decisão

**Fonte:** `alexteboul/diabetes-health-indicators-dataset` (BRFSS 2015, Kaggle).
Comando real para baixar os dados (requer conta Kaggle + `kaggle.json` configurado):

```bash
kaggle datasets download alexteboul/diabetes-health-indicators-dataset -p data --unzip
```

> Sem credencial/rede este notebook usa um **fallback sintético realista** (seed 42, n=5000,
> mesmas colunas do dataset original) para que figuras, HTML e testes sejam gerados offline.
> As relações foram calibradas para reproduzir os padrões conhecidos:
> pressão alta, colesterol alto, IMC e idade elevam o risco; atividade física protege.

Blocos: entendimento → preparação (balanceamento documentado, encoding binário) → EDA →
mineração (KMeans k=3, árvore de decisão + matriz de confusão, Apriori) → decisão (regra de triagem + prevenções).

In [1]:
import sys
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display
from src.common import load_csv, save_fig, df_to_html_table
from src.mining import run_kmeans, run_tree, confusion_summary
RNG = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 100
print("imports ok")

imports ok


## 1. Entendimento dos dados

Alvo: `Diabetes_binary` (0 = sem diabetes, 1 = pré/diabetes). Fatores: pressão alta (`HighBP`),
colesterol alto (`HighChol`), IMC (`BMI`), idade em 13 faixas (`Age`), atividade física (`PhysActivity`),
tabagismo, AVC, doença cardíaca, saúde geral (`GenHlth` 1–5), saúde mental/física (dias ruins em 30),
dificuldade de locomoção (`DiffWalk`), sexo e hábitos.

In [2]:
CANDIDATES = [
    "../data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    "../data/diabetes_binary_health_indicators_BRFSS2015.csv",
]
df = None
for c in CANDIDATES:
    try:
        df = load_csv(c)
        print("Carregado:", c)
        break
    except FileNotFoundError:
        continue
if df is None:
    print("Sem data/ local -> fallback SINTÉTICO realista (seed 42, n=5000).")
    print("Comando real: kaggle datasets download alexteboul/diabetes-health-indicators-dataset -p data --unzip")
    n = 5000
    age = RNG.integers(1, 14, n)
    highbp = (RNG.random(n) < 0.25 + 0.03 * age).astype(int)
    highchol = (RNG.random(n) < 0.25 + 0.025 * age).astype(int)
    cholcheck = (RNG.random(n) < 0.95).astype(int)
    bmi = np.clip(RNG.normal(28, 6, n), 12, 60).round(1)
    smoker = (RNG.random(n) < 0.44).astype(int)
    stroke = (RNG.random(n) < 0.04 + 0.004 * age).astype(int)
    heart = (RNG.random(n) < 0.08 + 0.006 * age).astype(int)
    physact = (RNG.random(n) < 0.75 - 0.015 * age).astype(int)
    fruits = (RNG.random(n) < 0.63).astype(int)
    veggies = (RNG.random(n) < 0.81).astype(int)
    hvyalc = (RNG.random(n) < 0.06).astype(int)
    anyhc = (RNG.random(n) < 0.89).astype(int)
    nodoc = (RNG.random(n) < 0.11).astype(int)
    genhlth = np.clip(RNG.integers(1, 6, n) + (RNG.random(n) < 0.2).astype(int), 1, 5)
    menthlth = np.clip(RNG.negative_binomial(2, 0.4, n), 0, 30)
    physhlth = np.clip(RNG.negative_binomial(2, 0.35, n), 0, 30)
    diffwalk = ((physhlth > 10) | (RNG.random(n) < 0.12 + 0.01 * age)).astype(int)
    sex = RNG.integers(0, 2, n)
    logit = (-3.2 + 0.9 * highbp + 0.8 * highchol + 0.06 * (bmi - 25)
             + 0.12 * age - 0.5 * physact + 0.25 * (genhlth - 3)
             + 0.4 * heart + 0.3 * stroke)
    p = 1 / (1 + np.exp(-logit))
    diab = (RNG.random(n) < p).astype(int)
    df = pd.DataFrame({
        "Diabetes_binary": diab, "HighBP": highbp, "HighChol": highchol,
        "CholCheck": cholcheck, "BMI": bmi, "Smoker": smoker, "Stroke": stroke,
        "HeartDiseaseorAttack": heart, "PhysActivity": physact, "Fruits": fruits,
        "Veggies": veggies, "HvyAlcoholConsump": hvyalc, "AnyHealthcare": anyhc,
        "NoDocbcCost": nodoc, "GenHlth": genhlth, "MentHlth": menthlth,
        "PhysHlth": physhlth, "DiffWalk": diffwalk, "Sex": sex, "Age": age,
    })
print("shape:", df.shape)
print("prevalência:", round(float(df['Diabetes_binary'].mean()), 4))
display(HTML(df_to_html_table(df, 10)))

Sem data/ local -> fallback SINTÉTICO realista (seed 42, n=5000).
Comando real: kaggle datasets download alexteboul/diabetes-health-indicators-dataset -p data --unzip
shape: (5000, 20)
prevalência: 0.2104


Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age
1,1,0,1,34.1,0,0,0,1,1,0,0,1,1,3,3,6,0,1,2
0,0,0,1,23.3,1,0,0,1,0,0,1,1,0,1,0,1,0,0,11
0,0,1,1,19.4,1,0,1,1,1,1,0,1,0,4,2,3,1,0,9
0,0,0,1,34.6,0,0,0,1,1,1,0,1,0,4,8,3,0,0,6
0,1,0,1,33.4,0,0,0,1,1,1,0,1,1,4,1,1,0,1,6
1,1,0,1,28.8,0,0,0,1,0,1,0,0,1,2,3,4,0,1,12
0,0,0,1,25.8,0,0,0,1,1,1,0,1,0,5,6,19,1,1,2
0,0,0,1,19.2,0,1,0,0,1,1,0,1,0,2,3,3,0,0,10
0,0,0,1,33.2,1,1,0,1,1,1,0,0,0,5,1,0,0,1,3
0,0,0,1,35.1,0,0,0,1,1,1,0,1,0,1,1,4,1,0,2


## 2. Preparação — balanceamento documentado e encoding binário

- **Encoding:** o dataset original já é binário (0/1) nas colunas indicadoras; `GenHlth` (1–5),
`Age` (1–13) e contagens de dias são mantidos como ordinais/numéricos — nenhum one-hot necessário.
- **Balanceamento (documentado):** a prevalência é minoritária (~15–20%), então a árvore é treinada
em amostra **balanceada por undersampling aleatório da classe majoritária** (seed 42).
KMeans e Apriori usam a base cheia (distribuição real preservada); apenas o classificador usa a base balanceada.

In [3]:
print("Distribuição original (proporção):")
print(df["Diabetes_binary"].value_counts(normalize=True).round(3).to_dict())
df0 = df[df.Diabetes_binary == 0]
df1 = df[df.Diabetes_binary == 1]
df0b = df0.sample(len(df1), random_state=42)
df_bal = pd.concat([df0b, df1]).sample(frac=1, random_state=42).reset_index(drop=True)
bal = pd.DataFrame({"classe": [0, 1], "n_antes": [len(df0), len(df1)],
                      "n_depois_undersampling": [len(df0b), len(df1)]})
display(HTML(df_to_html_table(bal)))
print("base balanceada:", df_bal.shape)

Distribuição original (proporção):
{0: 0.79, 1: 0.21}


classe,n_antes,n_depois_undersampling
0,3948,1052
1,1052,1052


base balanceada: (2104, 20)


## 3. EDA — prevalência por idade, pressão, colesterol e atividade + correlações

Quatro leituras de prevalência orientam a triagem; o heatmap mostra quais fatores andam juntos.

In [4]:
def prev_bar(col, labels, fname, title, xlabel):
    g = df.groupby(col)["Diabetes_binary"].mean()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([labels[i] for i in g.index], g.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Prevalência de diabetes")
    for i, v in enumerate(g.values):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    save_fig(fig, fname)
    plt.close(fig)
    display(HTML(f'<img src="../docs/assets/img/{fname}" alt="{title}">'))


prev_bar("Age", {i: f"F{i}" for i in range(1, 14)},
         "diabetes_prev_idade.png", "Prevalência de diabetes por faixa etária", "Faixa etária (1–13)")
prev_bar("HighBP", {0: "Sem pressao alta", 1: "Com pressao alta"},
         "diabetes_prev_pressao.png", "Prevalência de diabetes por pressão arterial", "Pressão alta")
prev_bar("HighChol", {0: "Sem colesterol alto", 1: "Com colesterol alto"},
         "diabetes_prev_colesterol.png", "Prevalência de diabetes por colesterol", "Colesterol alto")
prev_bar("PhysActivity", {0: "Sedentário", 1: "Ativo"},
         "diabetes_prev_atividade.png", "Prevalência de diabetes por atividade física", "Atividade física")

In [5]:
cols = ["Diabetes_binary", "HighBP", "HighChol", "BMI", "Smoker", "Stroke",
        "HeartDiseaseorAttack", "PhysActivity", "GenHlth", "PhysHlth",
        "DiffWalk", "Age"]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[cols].corr().round(2), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlação entre fatores e diabetes")
fig.tight_layout()
save_fig(fig, "diabetes_correlacao.png")
plt.close(fig)
display(HTML('<img src="../docs/assets/img/diabetes_correlacao.png" alt="Correlacoes">'))


## 4. Mineração com KMeans (k=3, perfis de risco)

KMeans com k=3 sobre atributos padronizados (IMC, idade, saúde geral, dias ruins,
pressão e colesterol) para descobrir 3 perfis de risco na população.

In [6]:
from sklearn.preprocessing import StandardScaler
feats = ["BMI", "Age", "GenHlth", "PhysHlth", "MentHlth", "HighBP", "HighChol"]
Xs = StandardScaler().fit_transform(df[feats])
labels, km = run_kmeans(pd.DataFrame(Xs, columns=feats), k=3, seed=42)
df["cluster"] = labels
prof = df.groupby("cluster")[feats + ["Diabetes_binary"]].mean().round(3).reset_index()
prof = prof.sort_values("Diabetes_binary").reset_index(drop=True)
display(HTML(df_to_html_table(prof)))
print("KMeans k=3 — tamanho dos clusters:", pd.Series(labels).value_counts().sort_index().to_dict())

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df["BMI"], df["Age"], c=df["cluster"], cmap="viridis", alpha=0.4, s=8)
ax.set_title("KMeans k=3 — clusters no plano IMC x idade")
ax.set_xlabel("BMI")
ax.set_ylabel("Faixa etária")
fig.colorbar(sc, ax=ax, label="cluster")
fig.tight_layout()
save_fig(fig, "diabetes_kmeans.png")
plt.close(fig)
display(HTML('<img src="../docs/assets/img/diabetes_kmeans.png" alt="Clusters KMeans">'))

cluster,BMI,Age,GenHlth,PhysHlth,MentHlth,HighBP,HighChol,Diabetes_binary
1,28.151,5.794,3.216,3.638,3.009,0.000,0.0,0.089
0,28.206,7.181,3.165,3.753,3.012,1.000,0.0,0.223
2,28.256,7.754,3.109,3.722,2.995,0.504,1.0,0.294


KMeans k=3 — tamanho dos clusters: {0: 1294, 1: 1590, 2: 2116}


## 5. Árvore de decisão + matriz de confusão

A árvore prevê `Diabetes_binary` na base balanceada (split 75/25 estratificado).
A matriz de confusão e a acurácia resumem o desempenho; a importância revela o top-5 de fatores.

In [7]:
from sklearn.model_selection import train_test_split
X = df_bal.drop(columns=["Diabetes_binary"])
y = df_bal["Diabetes_binary"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
tree = run_tree(Xtr, ytr)
pred = tree.predict(Xte)
s = confusion_summary(yte, pred)
print("Acurácia:", round(s["accuracy"], 3))
print("Matriz de confusão [[VN, FP], [FN, VP]]:", s["confusion_matrix"])
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
top5 = imp.head(5).reset_index()
top5.columns = ["fator", "importancia"]
top5["importancia"] = top5["importancia"].round(4)
display(HTML(df_to_html_table(top5)))
print("Top-5 fatores:", ", ".join(top5["fator"]))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(s["confusion_matrix"], annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Matriz de confusão — árvore de decisão")
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
fig.tight_layout()
save_fig(fig, "diabetes_confusao.png")
plt.close(fig)
display(HTML('<img src="../docs/assets/img/diabetes_confusao.png" alt="Matriz de confusao">'))

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top5["fator"][::-1], top5["importancia"][::-1])
ax.set_title("Top-5 fatores (importância da árvore)")
ax.set_xlabel("Importância")
fig.tight_layout()
save_fig(fig, "diabetes_importancia.png")
plt.close(fig)
display(HTML('<img src="../docs/assets/img/diabetes_importancia.png" alt="Top fatores">'))

Acurácia: 0.559
Matriz de confusão [[VN, FP], [FN, VP]]: [[147, 116], [116, 147]]


fator,importancia
BMI,0.2357
Age,0.1369
PhysHlth,0.1121
GenHlth,0.0887
MentHlth,0.0870


Top-5 fatores: BMI, Age, PhysHlth, GenHlth, MentHlth


## 6. Regras de associação (discretização + Apriori com mlxtend)

Fatores já binários dispensam discretização fina (variáveis contínuas como IMC/idade foram
excluídas da cesta). O Apriori (suporte ≥ 0.10, lift > 1.1) encontra co-ocorrências que viram regras.

In [8]:
from mlxtend.frequent_patterns import apriori, association_rules
basket = pd.DataFrame({
    "HighBP": df["HighBP"], "HighChol": df["HighChol"], "Smoker": df["Smoker"],
    "NoActivity": 1 - df["PhysActivity"], "Heart": df["HeartDiseaseorAttack"],
    "DiffWalk": df["DiffWalk"], "Diabetes": df["Diabetes_binary"],
})
freq = apriori(basket.astype(bool), min_support=0.10, use_colnames=True)
rules = association_rules(freq, metric="lift", min_threshold=1.1)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
top = rules.head(10)[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
top["antecedents"] = top["antecedents"].astype(str)
top["consequents"] = top["consequents"].astype(str)
top[["support", "confidence", "lift"]] = top[["support", "confidence", "lift"]].round(3)
print(f"{len(freq)} itemsets frequentes, {len(rules)} regras (lift>1.1)")
display(HTML(df_to_html_table(top, 10)))

19 itemsets frequentes, 6 regras (lift>1.1)


antecedents,consequents,support,confidence,lift
frozenset({'Diabetes'}),frozenset({'HighBP'}),0.141,0.671,1.422
frozenset({'HighBP'}),frozenset({'Diabetes'}),0.141,0.299,1.422
frozenset({'Diabetes'}),frozenset({'HighChol'}),0.124,0.591,1.397
frozenset({'HighChol'}),frozenset({'Diabetes'}),0.124,0.294,1.397
frozenset({'Diabetes'}),frozenset({'NoActivity'}),0.101,0.480,1.293
frozenset({'NoActivity'}),frozenset({'Diabetes'}),0.101,0.272,1.293


## 7. Decisão — regra de triagem e recomendações

**Regra de triagem (aplicável na atenção primária):**
SE (pressão alta = 1 E colesterol alto = 1 E faixa etária ≥ 9)
OU (IMC ≥ 30 E atividade física = 0)
ENTÃO encaminhar para rastreio prioritário de diabetes (glicemia de jejum + acompanhamento).

**Recomendações (3 prevenções priorizadas):**

1. Recomenda-se programa de atividade física para sedentários com IMC ≥ 30 — o grupo de maior prevalência na EDA.
2. Recomenda-se rastreio combinado pressão + colesterol em ≥ 55 anos (faixas 9–13), onde a prevalência dispara.
3. Recomenda-se acompanhamento de saúde geral (GenHlth ≥ 4) com avaliação de locomoção e dias de saúde ruim.

A amostra abaixo (200 linhas) replica o formato dos dados para auditoria.

In [9]:
sample = df.drop(columns=["cluster"]).sample(200, random_state=42)
outp = Path("../docs/assets/data/sample_diabetes.csv")
outp.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(outp, index=False)
print("sample:", sample.shape, "bytes:", outp.stat().st_size)
display(HTML(df_to_html_table(sample, 10)))

sample: (200, 20) bytes: 8884


Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age
0,1,1,1,25.0,1,0,0,0,0,1,0,1,0,4,11,4,1,1,4
0,0,0,1,38.7,1,0,0,1,1,1,0,1,0,4,1,6,0,0,5
1,1,1,1,27.7,0,0,0,0,1,1,0,1,0,4,2,6,0,1,5
0,0,1,1,21.4,0,0,0,1,0,1,0,1,1,2,0,3,0,0,13
0,1,1,1,30.5,0,0,0,1,1,0,0,1,1,2,5,0,0,0,8
0,0,0,1,18.9,0,0,0,1,0,1,0,1,1,5,0,10,0,1,11
0,0,1,1,36.4,1,0,0,0,0,1,0,1,0,1,17,2,0,0,11
0,1,1,1,25.6,0,0,0,1,0,1,1,1,0,4,4,0,0,0,1
0,0,1,1,27.7,1,0,0,1,1,1,0,1,0,3,1,9,0,1,8
0,1,1,1,23.8,0,0,0,1,0,1,0,1,0,5,3,7,0,1,4
